# 03 — Classical and Ensemble Models

Uses the same registry and pipelines as `train.py`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')

import pandas as pd, numpy as np
import config
from src import data_loader as dl

from src import train_ml, utils, evaluate as ev
from src.registry import REGISTRY, get_specs
utils.set_seeds()

## The split

Created in exactly one place, so no notebook can accidentally re-split.

In [ ]:
df = dl.load_raw()
splits = dl.make_splits(df)
print(splits.sizes)
splits.assert_disjoint()
print('positive rate — train %.4f val %.4f test %.4f' % (splits.y_train.mean(), splits.y_val.mean(), splits.y_test.mean()))

## The experiment registry

In [ ]:
for s in get_specs('full'):
    print('%-10s %-26s scale=%s tune=%s' % (s.family, s.name, s.needs_scaling, s.tune))

## Train a subset

The full sweep lives in `train.py`.

In [ ]:
NAMES = ['Dummy Baseline', 'Logistic Regression', 'Random Forest']
specs = [s for s in REGISTRY if s.name in NAMES]
utils.start_run(len(specs))

rows = []
for spec in specs:
    utils.step('Training ' + spec.name + '...')
    pipe = train_ml.build_pipeline(spec, splits.X_train)
    pipe.fit(splits.X_train, splits.y_train)
    m, _, _ = ev.evaluate_model(pipe, splits.X_val, splits.y_val)   # validation, not test
    rows.append({'Model': spec.name, **m})

pd.DataFrame(rows).round(4)

Evaluation here uses the **validation** set. The test set is reserved for the single final evaluation in `train.py`.

## Load the full results produced by train.py

In [ ]:
if config.RESULTS_CSV.exists():
    res = pd.read_csv(config.RESULTS_CSV)
    display(res[['Model','Family','Accuracy','Precision','Recall','F1','ROC-AUC']].round(4))
else:
    print('Run `python train.py --mode full` first.')